# DataPilot AI — Reference dataset exploration

Actual source: Online Retail II, Year 2010-2011. Original data is preserved. Recorded outputs below come from the application acceptance run; re-run cells to recalculate them. Installation is already complete.


In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / 'services').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from services.data_loader import DataLoader, inspect_dataset
from services.data_cleaner import prepare_data, analyze_data_quality, DataCleaner, CleaningOptions
from services.duckdb_service import DuckDBService
from ml.rfm import RFMAnalyzer
from ml.segmentation import SegmentationModel
from ml.anomaly_detection import AnomalyDetector
path = ROOT / 'data/raw/online_retail_II.xlsx'
raw = DataLoader().load_file(path, sheet_name='Year 2010-2011')


In [2]:
print(raw.shape)
print(raw.head())


(541910, 8)
  Invoice StockCode  ... Customer ID         Country
0  536365    85123A  ...     17850.0  United Kingdom
1  536365     71053  ...     17850.0  United Kingdom
2  536365    84406B  ...     17850.0  United Kingdom
3  536365    84029G  ...     17850.0  United Kingdom
4  536365    84029E  ...     17850.0  United Kingdom

[5 rows x 8 columns]


In [3]:
raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Description  540456 non-null  object        
 3   Quantity     541910 non-null  int64         
 4   InvoiceDate  541910 non-null  datetime64[us]
 5   Price        541910 non-null  float64       
 6   Customer ID  406830 non-null  float64       
 7   Country      541910 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [4]:
print(raw.describe())


            Quantity                 InvoiceDate          Price    Customer ID
count  541910.000000                      541910  541910.000000  406830.000000
mean        9.552234  2011-07-04 13:35:22.342307       4.611138   15287.684160
min    -80995.000000         2010-12-01 08:26:00  -11062.060000   12346.000000
25%         1.000000         2011-03-28 11:34:00       1.250000   13953.000000
50%         3.000000         2011-07-19 17:17:00       2.080000   15152.000000
75%        10.000000         2011-10-19 11:27:00       4.130000   16791.000000
max     80995.000000         2011-12-09 12:50:00   38970.000000   18287.000000
std       218.080957                         NaN      96.759765    1713.603074


In [5]:
print(raw.isnull().sum())
print("Duplicates:", raw.duplicated().sum())


Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64
Duplicates: 5268


In [6]:
profile = inspect_dataset(raw)
print(profile["schema"].to_string(index=False))
print(profile["numeric_columns"], profile["categorical_columns"], profile["datetime_columns"])


     Column           Type  Missing  Unique                                                                                                                                                     Examples
    Invoice         object        0   25900                                                                                                                       536365, 536366, 536368, 536367, 536369
  StockCode         object        0    4070                                                                                                                        85123A, 71053, 84406B, 84029G, 84029E
Description         object     1454    4223 WHITE HANGING HEART T-LIGHT HOLDER, WHITE METAL LANTERN, CREAM CUPID HEARTS COAT HANGER, KNITTED UNION FLAG HOT WATER BOTTLE, RED WOOLLY HOTTIE WHITE HEART.
   Quantity          int64        0     722                                                                                                                                               6, 8, 2, 3

## Quality before cleaning
Malformed values are reported. Cancellation and negative-value policies are explicit; missing customers are retained by default.

In [7]:
report = analyze_data_quality(raw)
print(report["issues"])
print(report["conversions"])


{'Duplicate rows': 5268, 'Missing cells': 136534, 'Negative quantities': 10624, 'Zero quantities': 0, 'Negative prices': 2, 'Zero prices': 2515, 'Malformed dates': 0, 'Cancelled invoices (rows)': 9288}
       Column   Before   After  Invalid values
0   InvoiceNo   object  string               0
1   StockCode   object  string               0
2  CustomerID  float64  string               0


## Feature engineering and DuckDB
Prepared data retains all original rows. Revenue is signed. No currency is assumed by the generic application.

In [8]:
prepared = prepare_data(raw)
db = DuckDBService()
db.register_dataframe(prepared)
result = db.execute_query('SELECT Country, SUM(Revenue) AS Revenue FROM sales GROUP BY Country ORDER BY Revenue DESC LIMIT 5')
print(result)
db.close()


          Country       Revenue
0  United Kingdom  8.187806e+06
1     Netherlands  2.846615e+05
2            EIRE  2.632768e+05
3         Germany  2.216982e+05
4          France  1.974219e+05


## Explicit duplicate-only cleaning
This policy keeps returns, cancellations, zero-price rows, and missing IDs for review.

In [9]:
cleaned, audit = DataCleaner().clean_with_report(raw, CleaningOptions(remove_duplicates=True))
print(audit)
cleaned.to_csv(ROOT / "data/processed/online_retail_2010_2011_cleaned.csv", index=False)


{'before_rows': 541910, 'after_rows': 536642, 'removed_rows': 5268, 'before_missing': 136534, 'after_missing': 136491, 'options': {'remove_duplicates': True, 'remove_cancelled': False, 'remove_nonpositive_quantity': False, 'remove_nonpositive_price': False, 'drop_missing_customer': False, 'drop_invalid_dates': False, 'missing_numeric': 'keep'}}


## Customer analytics and unusual transactions
RFM has its own documented eligibility rules. Cluster labels come from observed statistics, and anomalies are not fraud claims.

In [10]:
rfm = RFMAnalyzer().analyze(prepared)
customers, stats = SegmentationModel().segment(rfm)
print(stats)


   Cluster  ...                                          Rationale
0        0  ...  Recency > median; frequency <= median; monetar...
1        1  ...  Recency <= median; frequency > median; monetar...
2        2  ...  Recency <= median; frequency > median; monetar...
3        3  ...  Recency > median; frequency > median; monetary...

[4 rows x 8 columns]


In [11]:
anomalies = AnomalyDetector().detect(prepared)
print({k:v for k,v in anomalies.items() if k not in {"data", "anomalies"}})


{'count': 10838, 'rows_checked': 541910, 'excluded_rows': 0, 'features': ['Quantity', 'UnitPrice', 'Revenue']}
